# DC-Ops: Option B — Re-train YOLOv8n-seg with Improved Hyperparams

Improvements over initial training:
- 100 epochs (was 50)
- Higher mosaic augmentation
- Lower box/cls loss gains for noisy labels
- Warmup + cosine LR schedule
- Patience 25 (was 15)

Upload `dc_images.zip` and `dc_labels.zip` when prompted.

**Runtime → T4 GPU**

In [ ]:
!pip install -q ultralytics

import os, glob, zipfile, shutil, random, yaml
from pathlib import Path
from google.colab import files

# Upload data
images_exist = len(glob.glob('data/sample_images/*.*')) > 0
labels_exist = len(glob.glob('labels/*.txt')) > 0

if images_exist and labels_exist:
    print(f'Data loaded: {len(glob.glob("data/sample_images/*.*"))} images, {len(glob.glob("labels/*.txt"))} labels')
else:
    print('Upload dc_images.zip and dc_labels.zip...')
    uploaded = files.upload()
    if 'dc_images.zip' in uploaded:
        with zipfile.ZipFile('dc_images.zip', 'r') as z: z.extractall('.')
    if 'dc_labels.zip' in uploaded:
        with zipfile.ZipFile('dc_labels.zip', 'r') as z: z.extractall('.')
    print(f'Images: {len(glob.glob("data/sample_images/*.*"))}, Labels: {len(glob.glob("labels/*.txt"))}')

In [ ]:
# Prepare dataset
DC_CLASSES = [
    'server rack', 'compute tray', 'NVLink switch tray', 'network switch',
    'power shelf', 'cable', 'network port', 'LED indicator',
    'label', 'fan', 'cooling manifold', 'cable cartridge',
    'power connector', 'drive bay', 'management port', 'DPU',
]

for split in ['train', 'val']:
    for sub in ['images', 'labels']:
        os.makedirs(f'dataset_v2/{split}/{sub}', exist_ok=True)

images_dir = Path('data/sample_images')
labels_dir = Path('labels')

paired = []
for img in sorted(images_dir.iterdir()):
    if img.suffix.lower() in ('.jpg', '.jpeg', '.png', '.webp'):
        label = labels_dir / f'{img.stem}.txt'
        if label.exists() and label.stat().st_size > 0:
            paired.append((img, label))

random.seed(42)
random.shuffle(paired)
split_idx = int(len(paired) * 0.85)

for img, lbl in paired[:split_idx]:
    shutil.copy2(img, f'dataset_v2/train/images/{img.name}')
    shutil.copy2(lbl, f'dataset_v2/train/labels/{img.stem}.txt')
for img, lbl in paired[split_idx:]:
    shutil.copy2(img, f'dataset_v2/val/images/{img.name}')
    shutil.copy2(lbl, f'dataset_v2/val/labels/{img.stem}.txt')

config = {
    'path': os.path.abspath('dataset_v2'),
    'train': 'train/images',
    'val': 'val/images',
    'names': {i: n for i, n in enumerate(DC_CLASSES)},
}
with open('dataset_v2/dataset.yaml', 'w') as f:
    yaml.dump(config, f, default_flow_style=False)

print(f'Dataset: {split_idx} train, {len(paired)-split_idx} val')

In [ ]:
# Train with improved hyperparameters
from ultralytics import YOLO

model = YOLO('yolov8n-seg.pt')

results = model.train(
    data='dataset_v2/dataset.yaml',
    epochs=100,
    batch=16,
    imgsz=640,
    project='runs_v2',
    name='dc_ops_seg_v2',
    exist_ok=True,
    device=0,
    patience=25,
    save=True,
    plots=True,
    verbose=True,
    # Improved hyperparams
    lr0=0.005,
    lrf=0.01,
    warmup_epochs=5,
    warmup_momentum=0.5,
    mosaic=1.0,
    mixup=0.1,
    copy_paste=0.1,
    degrees=10.0,
    translate=0.2,
    scale=0.5,
    fliplr=0.5,
    hsv_h=0.015,
    hsv_s=0.4,
    hsv_v=0.4,
)

In [ ]:
# Validate
import glob as g
best_path = g.glob('runs_v2/**/best.pt', recursive=True)[0]
print(f'Best model: {best_path}')

best = YOLO(best_path)
metrics = best.val()

print(f'\nmAP50 (box):  {metrics.box.map50:.3f}')
print(f'mAP50 (mask): {metrics.seg.map50:.3f}')
print(f'\nPer-class mAP50 (mask):')
for i, name in enumerate(DC_CLASSES):
    if i < len(metrics.seg.ap50):
        print(f'  {name:25s} {metrics.seg.ap50[i]:.3f}')

In [ ]:
# Download improved model
import shutil
shutil.copy2(best_path, 'dc_ops_yolov8n_seg_v2.pt')
print(f'Size: {os.path.getsize("dc_ops_yolov8n_seg_v2.pt") / 1e6:.1f} MB')

from google.colab import files
files.download('dc_ops_yolov8n_seg_v2.pt')